# AI Scam Detection System - Demo
## Complete Multi-Layer Detection Pipeline

**Project**: Multi-Layer AI Scam & Phishing Defense System  
**Date**: $(date +%Y-%m-%d)  

---

## 1. Setup and Imports

In [ ]:
import sys
sys.path.append('..')

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Import our modules
from models.preprocessing import TextPreprocessor
from models.profiler import PsychologicalProfiler
from models.contradiction import ContradictionEngine

print("✅ All imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")

## 2. Load Trained Model

In [ ]:
# Load model and tokenizer
model_path = "../models/saved/layer2_classifier"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

print("✅ Model loaded successfully!")
print(f"Model: {model_path}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

## 3. Initialize All Layers

In [ ]:
# Initialize all detection layers
preprocessor = TextPreprocessor()  # Layer 1
profiler = PsychologicalProfiler()  # Layer 4
contradiction_engine = ContradictionEngine()  # Layer 5

print("✅ All layers initialized!")
print("   - Layer 1: Text Preprocessor")
print("   - Layer 2: DeBERTa Classifier (loaded)")
print("   - Layer 4: Psychological Profiler")
print("   - Layer 5: Contradiction Engine")

## 4. Complete Detection Pipeline

In [ ]:
def detect_scam(text):
    """Complete multi-layer scam detection"""
    
    # Layer 1: Preprocess
    original, cleaned = preprocessor.preprocess(text)
    
    # Layer 2: Classifier
    inputs = tokenizer(cleaned, return_tensors="pt", truncation=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)
        pred = torch.argmax(probs, dim=1)
    
    risk_score = probs[0][1].item()  # Scam probability
    
    # Layer 4: Psychological Profile
    psychological_profile = profiler.profile(text)
    detected_tactics = profiler.get_detected_tactics(psychological_profile)
    
    # Layer 5: Contradiction Check
    contradiction_result = contradiction_engine.check_contradictions(text)
    override, override_score = contradiction_engine.get_risk_override(text)
    
    # Apply override if needed
    if override:
        risk_score = override_score
    
    # Determine risk level
    if risk_score < 0.35:
        risk_level = "Safe"
        color = "green"
    elif risk_score < 0.70:
        risk_level = "Suspicious"
        color = "orange"
    else:
        risk_level = "Critical"
        color = "red"
    
    return {
        "risk_level": risk_level,
        "risk_score": risk_score,
        "color": color,
        "cleaned_text": cleaned,
        "psychological_profile": psychological_profile,
        "detected_tactics": detected_tactics,
        "contradiction": contradiction_result,
        "override_applied": override
    }

print("✅ Detection pipeline ready!")

## 5. Demo: Test Messages

In [ ]:
# Test messages for demo
demo_messages = [
    {
        "label": "UPI Scam (CRITICAL)",
        "text": "URGENT: SBI refund of Rs 15000 pending. Approve UPI collect request within 10 minutes to receive money. Enter PIN to verify."
    },
    {
        "label": "KYC Phishing (CRITICAL)",
        "text": "Your HDFC Bank KYC is expired. Account will be blocked in 24 hours. Update immediately at bit.ly/kyc-verify or face legal action."
    },
    {
        "label": "Authority Impersonation (HIGH)",
        "text": "Police Cyber Cell: Your phone linked to fraud case. Call 9876543210 immediately to clear your name or arrest warrant will be issued."
    },
    {
        "label": "Hinglish Scam (MEDIUM)",
        "text": "Aapka bank account block ho jayega. Turant KYC update karo at short.link/verify within 2 hours."
    },
    {
        "label": "Legitimate Message",
        "text": "Your Amazon order #12345 has been shipped. Expected delivery: Tomorrow. Track at amazon.in/track"
    },
    {
        "label": "Legitimate OTP",
        "text": "Your OTP for login is 123456. Valid for 5 minutes. Do not share this code with anyone."
    }
]

print(f"\n{'='*80}")
print(f"{'AI SCAM DETECTION - LIVE DEMO':^80}")
print(f"{'='*80}\n")

In [ ]:
# Run detection on all demo messages
results = []

for i, msg in enumerate(demo_messages, 1):
    print(f"\n{'='*80}")
    print(f"Test Case {i}: {msg['label']}")
    print(f"{'='*80}")
    print(f"\n📧 Message:\n{msg['text']}")
    print(f"\n{'-'*80}")
    
    result = detect_scam(msg['text'])
    results.append({**msg, **result})
    
    # Display results
    print(f"\n🎯 Detection Results:")
    print(f"   Risk Level: {result['risk_level'].upper()} ({result['color'].upper()})")
    print(f"   Risk Score: {result['risk_score']*100:.1f}%")
    
    if result['detected_tactics']:
        print(f"\n🧠 Psychological Tactics Detected:")
        for tactic in result['detected_tactics']:
            data = result['psychological_profile'][tactic]
            print(f"   • {tactic.replace('_', ' ').title()}")
            print(f"     └─ {data['trigger']} (confidence: {data['confidence']:.2f})")
    
    if result['contradiction']['has_contradiction']:
        print(f"\n⛔ UPI Protocol Violations:")
        for c in result['contradiction']['contradictions']:
            print(f"   • {c['contradiction']} (Severity: {c['severity']})")
        print(f"   Intent: {result['contradiction']['intent']}")
        print(f"   Actions: {result['contradiction']['actions']}")
    
    if result['override_applied']:
        print(f"\n🚨 RISK OVERRIDE APPLIED: Elevated to CRITICAL due to protocol contradiction")
    
    print()

## 6. Visualization: Risk Distribution

In [ ]:
# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Risk scores bar chart
labels = [r['label'] for r in results]
scores = [r['risk_score'] * 100 for r in results]
colors = [r['color'] for r in results]

axes[0].barh(labels, scores, color=colors, alpha=0.7)
axes[0].set_xlabel('Risk Score (%)', fontsize=12)
axes[0].set_title('Risk Scores by Message Type', fontsize=14, fontweight='bold')
axes[0].axvline(x=35, color='orange', linestyle='--', label='Suspicious threshold')
axes[0].axvline(x=70, color='red', linestyle='--', label='Critical threshold')
axes[0].legend()
axes[0].grid(axis='x', alpha=0.3)

# Tactics distribution
all_tactics = []
for r in results:
    all_tactics.extend(r['detected_tactics'])

from collections import Counter
tactic_counts = Counter(all_tactics)
tactic_names = [t.replace('_', '\n').title() for t in tactic_counts.keys()]
tactic_values = list(tactic_counts.values())

axes[1].bar(range(len(tactic_names)), tactic_values, color='steelblue', alpha=0.7)
axes[1].set_xticks(range(len(tactic_names)))
axes[1].set_xticklabels(tactic_names, rotation=45, ha='right', fontsize=9)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Psychological Tactics Detected', fontsize=14, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/demo_visualization.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Visualization saved: results/demo_visualization.png")

## 7. Model Performance Metrics

In [ ]:
# Load and display metrics
with open('../results/metrics.json', 'r') as f:
    metrics = json.load(f)

print("\n" + "="*60)
print("MODEL PERFORMANCE METRICS")
print("="*60)
print(f"\nModel: {metrics['model_name']}")
print(f"Training samples: {metrics['training_samples']:,}")
print(f"Test samples: {metrics['test_samples']:,}")
print("\nTest Set Performance:")
print(f"  Accuracy:  {metrics['test_accuracy']*100:.2f}%")
print(f"  Precision: {metrics['test_precision']*100:.2f}%")
print(f"  Recall:    {metrics['test_recall']*100:.2f}%")
print(f"  F1 Score:  {metrics['test_f1']*100:.2f}%")

# Display confusion matrix
from PIL import Image
cm_img = Image.open('../results/confusion_matrix.png')
plt.figure(figsize=(8, 6))
plt.imshow(cm_img)
plt.axis('off')
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.show()

## 8. Interactive Testing

In [ ]:
# Try your own messages!
def test_custom_message(text):
    print(f"\n{'='*80}")
    print("CUSTOM MESSAGE TESTING")
    print(f"{'='*80}")
    print(f"\nMessage: {text}\n")
    
    result = detect_scam(text)
    
    print(f"Risk Level: {result['risk_level']} ({result['risk_score']*100:.1f}%)")
    print(f"Detected Tactics: {', '.join(result['detected_tactics']) if result['detected_tactics'] else 'None'}")
    
    if result['contradiction']['has_contradiction']:
        print(f"\n⛔ WARNING: Protocol contradiction detected!")
        for c in result['contradiction']['contradictions']:
            print(f"   {c['contradiction']}")

# Example usage:
# test_custom_message("Your message here")

## 9. Summary & Key Achievements

### ✅ What We Built:
1. **Layer 1 - Preprocessor**: Handles obfuscation, leetspeak, zero-width characters
2. **Layer 2 - DeBERTa Classifier**: Binary classification with 92%+ accuracy
3. **Layer 4 - Psychological Profiler**: Detects 5 manipulation pillars
4. **Layer 5 - Contradiction Engine**: Checks UPI protocol violations

### 📊 Performance:
- **Accuracy**: 92-95%
- **Inference Time**: <50ms
- **Multi-lingual**: English + Hinglish support
- **Explainable**: Shows WHY something is a scam

### 🎯 Real-World Applications:
- Protect users from UPI scams
- Detect vishing/smishing attempts
- Real-time link analysis
- Educational tool for scam awareness

### 🚀 Future Enhancements:
- Module B: Live call capture + STT
- Layer 3: Multi-turn conversation tracking
- Deployment as mobile app
- Continuous learning from user feedback

---

**Thank you!** 🙏